# Exploration de données statistiques avec Pandas et Seaborn

Notebook de l'article [Exploration de données statistiques avec Pandas et Seaborn](https://moncoachdata.com/blog/exploration-de-donnees-statistiques/) (MonCoachData). Exécute les cellules dans l'ordre.

Données : *120 ans d'histoire olympique : athlètes et résultats*, fichier `athlete_events.csv` ([page Kaggle](https://www.kaggle.com/heesoo37/120-years-of-olympic-history-athletes-and-results)). Le code est testé avec Pandas 3 et Seaborn 0.13, et il fonctionne aussi avec Pandas 2 (version de Google Colab).

Dans cet article, tu vas réaliser une **exploration de données statistiques**. J'utilise la bibliothèque [Pandas](https://moncoachdata.com/blog/analyse-exploratoire-de-donnees/) pour l'analyse de données et la bibliothèque [Seaborn](https://seaborn.pydata.org/tutorial.html) pour la visualisation. Seaborn est un formidable outil de visualisation d'un point de vue esthétique.

Seaborn s'appuie sur [Matplotlib](https://matplotlib.org/), mais il configure les graphiques avec des styles par défaut qui les rendent beaucoup plus beaux. On va examiner un jeu de données, et je vais t'expliquer comment utiliser différentes caractéristiques. Qui sait, peut-être en tirerons-nous des hypothèses ou des conclusions intéressantes !

Face à un problème de Data Science, il faut d'abord se plonger dans le jeu de données (ou dataset) : étudier ses propriétés, ses distributions, etc.

**En bref :** dans le dataset « 120 ans d'histoire olympique » (271 116 participations, 15 colonnes), seules 15 % des participations donnent lieu à une médaille : environ 28 000 athlètes sur près de 135 000 en ont gagné au moins une. La participation des femmes, presque nulle au départ, se rapproche peu à peu de celle des hommes, mais aucune année du dataset ne compte plus de femmes que d'hommes. L'analyse utilise Pandas et Seaborn.

## Quel dataset utilise-t-on pour explorer les Jeux olympiques ?

Pour cette analyse, j'utilise le dataset **« 120 ans d'histoire olympique : athlètes et résultats »**. Tu peux le télécharger et lire sa description sur [Kaggle](https://www.kaggle.com/heesoo37/120-years-of-olympic-history-athletes-and-results).

Je l'ai téléchargé gratuitement sur Kaggle, le site de compétitions de Machine Learning. Il est génial si tu as besoin d'un jeu de données pour essayer un nouvel algorithme, ou tout simplement pour t'amuser un peu.

Je n'utilise que le fichier CSV `athlete_events`. Il répertorie tous les compétiteurs de chaque compétition olympique, avec leur pays, l'éventuelle médaille obtenue, etc.

Fait intéressant : la colonne des médailles est vide à 85 %. Seulement 15 % environ des participations aux Jeux olympiques donnent donc lieu à une médaille. Certains athlètes en gagnent même plusieurs : il y a donc peu de médaillés parmi les participants. Cela montre la difficulté de gagner une médaille, et le mérite de ces fantastiques athlètes !

## À quoi ressemble le jeu de données des Jeux olympiques ?

Avant de chercher à recueillir des informations, j'aime bien avoir une meilleure connaissance du jeu de données.

Combien de données sont manquantes ? Combien y a-t-il de colonnes ? Ce sont les premières questions que je me suis posées.

J'utilise pour cette analyse un notebook Jupyter, parfait pour explorer rapidement un jeu de données. Je te donne un extrait de code pour chaque ligne pertinente, pour que tu puisses suivre.

Je commence par charger les données avec la bibliothèque [Pandas](https://pandas.pydata.org/docs/) et vérifier leur dimension avec l'attribut `shape` :

## Récupérer les données

La cellule suivante télécharge `athlete_events.csv` (271 116 lignes, 15 colonnes) depuis le dépôt GitHub de MonCoachData (copie du fichier compilé par Randi Griffin) : pas de compte ni de jeton nécessaire. Si le fichier est déjà à côté du notebook, rien n'est téléchargé.

In [ ]:
from pathlib import Path
import urllib.request

FICHIER = 'athlete_events.csv'
URL = 'https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/exploration-de-donnees-statistiques/athlete_events.csv'

if not Path(FICHIER).exists():
    urllib.request.urlretrieve(URL, FICHIER)

print(FICHIER, 'prêt :', round(Path(FICHIER).stat().st_size / 1e6, 1), 'Mo')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

df = pd.read_csv('athlete_events.csv')
print(df.shape)
df.head()

Le jeu de données a 15 colonnes et 271 116 lignes, soit plus de 270 000 participations ! Je me demande combien d'athlètes différents il y a en réalité, et combien ont réellement gagné des médailles.

Pour commencer, je liste les colonnes du DataFrame. Cela donne de nombreuses caractéristiques intéressantes :

In [ ]:
print(df.columns.tolist())

### Que peut-on analyser avec ce jeu de données ?

On pourrait regarder la taille (colonne `Height`) et le poids (colonne `Weight`) moyens des athlètes olympiques par sport.

On pourrait aussi voir la distribution de ces deux variables en fonction du sexe, ou le nombre de médailles de chaque pays en série chronologique, pour voir l'ascension et le déclin des pays au cours du XXe siècle.

Les possibilités sont infinies ! Mais abordons d'abord les problèmes qui fâchent : notre jeu de données est-il complet ?

In [ ]:
# Pourcentage de valeurs manquantes par colonne
missing_percent = df.isnull().mean() * 100
print(missing_percent.round(2))

La méthode `isnull()` renvoie un tableau de booléens (`True` pour une valeur manquante). Comme `True` vaut 1 et `False` vaut 0, `mean()` donne directement la proportion de valeurs manquantes de chaque colonne. Il suffit de multiplier par 100.

Quatre colonnes sont incomplètes : `Age`, `Height`, `Weight` et `Medal`. Note les 15 % d'athlètes qui ont obtenu au moins une médaille dans la colonne `Medal`.

La colonne `Medal` est vide chaque fois qu'un athlète ne remporte pas de médaille. On s'attend donc à ce qu'elle soit peu remplie. En revanche, pour `Weight`, `Height` et `Age`, il y a pas mal de valeurs manquantes.

J'ai essayé de filtrer les lignes selon différentes années, mais les valeurs manquantes semblent constantes dans le temps. Je pense donc que certains pays ne fournissent pas ces données sur leurs athlètes.

## Combien d'athlètes ont réellement gagné une médaille ?

La première question que je me suis posée : combien de personnes différentes ont réellement remporté une médaille ? Le code suivant répond à cette question :

In [ ]:
total_rows = df.shape[0]
unique_athletes = df['Name'].nunique()
medal_winners = df.loc[df['Medal'].notna(), 'Name'].nunique()

print(total_rows, unique_athletes, medal_winners)

Près de 135 000 personnes différentes ont participé aux Jeux olympiques au cours des 120 dernières années, mais seulement un peu plus de 28 000 ont remporté au moins une médaille.

C'est à peu près un athlète sur cinq, ce qui n'est pas si mal. Mais c'est moins optimiste si tu considères que beaucoup d'athlètes concourent dans plusieurs épreuves.

Combien de médailles ont été réellement gagnées au cours de ces 120 années ?

In [ ]:
# Distribution des médailles
medals = df.loc[df['Medal'].notna(), 'Medal']
print(medals.value_counts())

# Nombre total de médailles (or, argent et bronze)
print(len(medals))

Sans surprise, la distribution des médailles est presque uniforme : il y a quasiment autant de médailles d'or, d'argent et de bronze.

Mais près de 40 000 médailles ont été attribuées pour environ 28 000 médaillés. Si tu fais partie de ce top 20 % des athlètes qui remportent une médaille, tu en gagnes donc plus d'une en moyenne.

### Quels pays ont gagné le plus de médailles ?

Pour l'obtenir, on exécute les extraits de code suivants :

In [ ]:
team_medal_count = (
    df.groupby(['Team', 'Medal'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
# team_medal_count.head(40) pour voir les 40 premières lignes

def get_country_stats(country):
    return team_medal_count[team_medal_count['Team'] == country]

# get_country_stats('France') pour obtenir les médailles d'un pays

In [ ]:
team_medal_count.head(10)

In [ ]:
get_country_stats('France')

Avec cette fonction, tu obtiens le nombre de médailles de chaque type gagnées par un pays donné. En affichant les premières lignes du DataFrame, tu vois les pays qui ont le plus de médailles.

Il est intéressant de noter que l'Union soviétique est toujours le deuxième pays à avoir remporté le plus de médailles depuis le début des Jeux, même si elle n'existe plus depuis 1991.

La première place est réservée aux États-Unis, toutes catégories confondues, et la troisième à l'Allemagne. J'ai aussi regardé la France (ma nation), qui se classe 6e au classement des médailles d'or.

Écrire un script pour obtenir les années auxquelles un pays a participé aux Jeux olympiques est un bon exercice pour toi !

## Quelle est la part des femmes aux Jeux olympiques ?

Autre question : quelle était la part des femmes aux Jeux olympiques au cours du siècle ? Le code suivant répond à la question :

In [ ]:
unique_women = df.loc[df['Sex'] == 'F', 'Name'].nunique()
unique_men = df.loc[df['Sex'] == 'M', 'Name'].nunique()
women_medals = df.loc[df['Sex'] == 'F', 'Medal'].count()
men_medals = df.loc[df['Sex'] == 'M', 'Medal'].count()

print(unique_women, unique_men, women_medals, men_medals)

# Première année avec une participante
print(df.loc[df['Sex'] == 'F', 'Year'].min())

J'ai été surpris de voir des femmes participer aux Jeux dès 1900. Au total, il y a environ trois hommes pour une femme. Surpris par cette participation féminine en 1900, j'ai décidé de vérifier leur nombre au fil du temps. J'ai enfin pu utiliser Seaborn !

In [ ]:
import matplotlib.pyplot as plt

# Nombre de participantes par année
f_year_count = df[df['Sex'] == 'F'].groupby('Year')['Name'].count()

sns.scatterplot(x=f_year_count.index, y=f_year_count.values)
plt.xlabel('Année')
plt.ylabel('Nombre de participantes')
plt.show()

La participation féminine a augmenté très rapidement ces dernières décennies, en partant de presque zéro pour atteindre des milliers de participantes.

### La participation des femmes augmente-t-elle plus vite que celle des hommes ?

Mais leur participation augmente-t-elle plus vite que celle des hommes ? Ou est-ce juste une question de population mondiale ? Pour le savoir, j'ai créé ce deuxième graphique :

In [ ]:
f_year_count = df[df['Sex'] == 'F'].groupby('Year')['Name'].count()
m_year_count = df[df['Sex'] == 'M'].groupby('Year')['Name'].count()

sns.scatterplot(x=m_year_count.index, y=m_year_count.values, label='Hommes')
sns.scatterplot(x=f_year_count.index, y=f_year_count.values, label='Femmes')
plt.xlabel('Année')
plt.ylabel('Nombre de participants')
plt.show()

Cette fois, une tendance se dessine clairement : la participation des femmes se rapproche de celle des hommes au fil du temps !

Autre chose amusante : regarde ces petits points en bas à droite du graphique. Je pense qu'il s'agit des Jeux olympiques d'hiver, qui comptent moins d'athlètes. Quoi qu'il en soit, la situation semble plutôt optimiste pour la représentation féminine, même si aucune année de ce dataset ne compte plus de femmes que d'hommes.

## Que montrent la taille et le poids des athlètes ?

J'ai passé beaucoup de temps à regarder les graphiques de taille et de poids, mais je n'en ai tiré aucune conclusion vraiment intéressante :

- les deux variables sont clairement distribuées de façon normale pour la plupart des sports ;
- les hommes sont toujours plus lourds et plus grands que les femmes dans tous les sports que j'ai vérifiés ;
- la seule variation intéressante semble être la répartition de chaque sexe en fonction du sport.

Je n'ai pas assez approfondi le regroupement par sport, il est donc possible d'en tirer quelques enseignements. À toi de jouer : essaie d'analyser `Weight` et `Height` par sport avec `groupby()` et Seaborn.

La deuxième partie sur l'extraction de données sur les Jeux olympiques est disponible : elle traite de [la taille et du poids des athlètes en fonction du sport pratiqué](https://moncoachdata.com/blog/extraction-dinformations-avec-pandas-et-seaborn).

Pour aller plus loin, consulte aussi notre guide des [astuces Pandas en Python](https://moncoachdata.com/blog/10-astuces-pandas-python/) et celui des [21 astuces et concepts Python pour améliorer ton code](https://moncoachdata.com/blog/21-astuces-et-concepts-python-pour-ameliorer-ton-code/).